In [13]:
!pip install -q --ignore-requires-python git+https://github.com/structuredllm/syncode.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [14]:
# i am defining the grammar that the model should recogonise according to LARK style grammar syntax in the SynCode Repo
gram="""
start: "Operands: " NUM " and " NUM " Operator: " OP
NUM: /-?[0-9]+\\.[0-9]+/
OP: "+" | "-" | "*" | "/"
"""

In [15]:
# some few shot examples for the model to understand the format of the input and how we want the questions answered
few_shot="""
Query: What is -27.65 plus 19.7?
Operands: -27.65 and 19.7 Operator: +

Query: What is 372. minus 22.89?
Operands: 372.0 and 22.89 Operator: -

Query: What is 327.5 multiplied by 11.?
Operands: 327.5 and 11.0 Operator: *

Query: What is 100. divided by 8.?
Operands: 100.0 and 8.0 Operator: /

"""

In [16]:
# now i am loading the model microsoft phi 2 inside syncode so that it follows our grammar
import re
from syncode import Syncode

our_model = Syncode(model="microsoft/phi-2", grammar=gram, parse_output_only=True, max_new_tokens=40)

[2026-10-28 02:42:54,367-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16
[2026-10-28 02:42:54,701-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[2026-10-28 02:43:17,457-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_4312106961_50257.pkl exist: False
[2026-10-28 02:43:17,458-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_4312106961_50257.pkl.
[2026-10-28 02:43:17,972-syncode.mask_store.fsm_set] - 5 FSMs with 41 states initialized in 0.00 seconds
[2026-10-28 02:43:17,973-syncode.mask_store.mask_store] - Ignore whitespace is False
[2026-10-28 02:43:17,978-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 25 to 5


100%|██████████| 41/41 [00:07<00:00,  5.33it/s]

[2026-10-28 02:43:25,683-syncode.mask_store.mask_store] - Time taken to create mask store: 8.22 seconds


In [17]:
def run_syncode(textual_query):
    prompt = few_shot + "Query: " + textual_query.strip()
    outp = our_model.infer(prompt)[0]
    parts = outp.split()
    num1 = float(parts[1])
    num2 = float(parts[3])
    op = parts[5]

    if op == "+":
        res = num1 + num2
    elif op == "-":
        res = num1 - num2
    elif op == "*":
        res = num1 * num2
    else:
        res = num1 / num2

    return ([num1, num2], op, round(res, 6))

In [19]:
print(run_syncode("What is 327. multiplied by 11.0?"))
print(run_syncode("What is 45.1 plus 23.54?"))
print(run_syncode("What is 120.4 divided by 4.0?"))

([327.0, 11.0], '*', 3597.0)
([45.1, 23.54], '+', 68.64)
([120.4, 4.0], '/', 30.1)


In [20]:
print(run_syncode("What is 8.5 minus 3.25?"))
print(run_syncode("What is 50.0 divided by 8.0?"))
print(run_syncode("What is 12. multiplied by 3.5?"))

([8.5, 3.25], '-', 5.25)
([50.0, 8.0], '/', 6.25)
([12.0, 3.5], '*', 42.0)


In [21]:
print(run_syncode("What is 45.1 added with 23.54?"))

([45.1, 23.54], '+', 68.64)


In [22]:
print(run_syncode("What is 327. times 11.0?"))
print(run_syncode("What is 45.1 and 23.54 added together?"))

([327.0, 11.0], '*', 3597.0)
([45.1, 23.54], '+', 68.64)


In [23]:
print(run_syncode("What is difference between 8.5 and 3.25?"))

([8.5, 3.25], '-', 5.25)


In [24]:
print(run_syncode("subtract 3.0 from 10.0"))

([10.0, 3.0], '-', 7.0)


In [18]:
print(run_syncode("subtract -3.0 from 10.0"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


([10.0, -3.0], '-', 13.0)


In [25]:
print(run_syncode("What is 45.1 plus -23.54?"))

([45.1, -23.54], '+', 21.56)
